In [1]:
import pandas as pd
from pathlib import Path

files = sorted([f.name for f in Path("data").iterdir() if f.is_file()])
df = pd.concat([
    pd.read_csv(f"data/{file}") for file in files
])

df

,open_time,open,high,low,close,volume,close_time,quote_volume,count,taker_buy_volume,taker_buy_quote_volume,ignore
0,1577836800000,129.12,129.12,128.76,128.81,2438.197,1577837099999,3.143663e+05,151,1127.136,1.453314e+05,0
1,1577837100000,128.78,128.94,128.65,128.87,2126.484,1577837399999,2.738965e+05,164,1377.615,1.774575e+05,0
2,1577837400000,128.90,128.92,128.77,128.87,888.920,1577837699999,1.145148e+05,95,487.341,6.277934e+04,0
3,1577837700000,128.88,128.99,128.77,128.82,1570.849,1577837999999,2.024439e+05,113,688.471,8.873060e+04,0
4,1577838000000,128.81,128.85,128.66,128.67,621.585,1577838299999,8.003575e+04,57,360.833,4.645903e+04,0
...,...,...,...,...,...,...,...,...,...,...,...,...
8635,1782862500000,1571.00,1571.00,1569.23,1569.24,2978.315,1782862799999,4.676017e+06,6135,1627.408,2.555045e+06,0
8636,1782862800000,1569.23,1570.27,1568.61,1570.14,3609.242,1782863099999,5.664026e+06,6748,1773.936,2.783775e+06,0
8637,1782863100000,1570.15,1572.21,1569.67,1570.39,14444.464,1782863399999,2.269243e+07,14558,9666.018,1.518565e+07,0
8638,1782863400000,1570.39,1571.14,1569.60,1571.11,6822.251,1782863699999,1.071485e+07,10540,4129.600,6.486000e+06,0


In [2]:
import numpy as np

INTERVAL_MS = 5 * 60 * 1000  # 5 minutes in ms

# Assert continuous open time
diff = np.diff(df["open_time"].to_numpy())
assert np.all(diff == INTERVAL_MS), (
    f"Found {np.sum(diff != INTERVAL_MS)} discontinuities.\n"
    f"First few indices: {np.where(diff != INTERVAL_MS)[0][:10]}"
)

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class CryptoDeltaDataset(Dataset):
    def __init__(self, df: pd.DataFrame, past_window: int, future_window: int, eps=1e-9):
        self.past_window = past_window
        self.future_window = future_window
        self.eps = eps

        self.df = df.reset_index(drop=True)
        self.features = ["open", "high", "low", "close", "volume", "count"]

        values = torch.tensor(self.df[self.features].values)
        diff = values[1:] - values[:-1]
        self.deltas = torch.log2(diff.abs() + 1) * torch.sign(diff)

        self.min_deltas = self.deltas.min(dim=0).values.unsqueeze(0)
        self.max_deltas = self.deltas.max(dim=0).values.unsqueeze(0)

    def __len__(self):
        return len(self.deltas) - self.past_window - self.future_window

    def __getitem__(self, idx):
        delta = self.deltas[idx:idx + self.future_window + self.past_window]

        # Normalize to [0, 1]
        normalized = ((delta - self.min_deltas) / (self.max_deltas - self.min_deltas + self.eps)).to(torch.float32)

        return (normalized[:self.past_window], normalized[self.past_window:])

    def collate_fn(self, batch):
        return [torch.stack(d, dim=0) for d in zip(*batch)]

dataset = CryptoDeltaDataset(df, past_window=12*24*30, future_window=12*24*30)
dataloader = DataLoader(dataset=dataset, batch_size=2, collate_fn=dataset.collate_fn)

next(iter(dataloader))[0]

tensor([[[0.4855, 0.4723, 0.4859, 0.5127, 0.3032, 0.5998],
         [0.5176, 0.4857, 0.5029, 0.5081, 0.2538, 0.3427],
         [0.5071, 0.4937, 0.4940, 0.5043, 0.7443, 0.6112],
         ...,
         [0.5459, 0.5300, 0.5302, 0.5325, 0.2975, 0.3783],
         [0.5324, 0.4884, 0.5127, 0.5089, 0.6873, 0.3217],
         [0.5110, 0.4848, 0.4760, 0.5020, 0.7641, 0.7030]],

        [[0.5176, 0.4857, 0.5029, 0.5081, 0.2538, 0.3427],
         [0.5071, 0.4937, 0.4940, 0.5043, 0.7443, 0.6112],
         [0.5033, 0.4755, 0.4859, 0.4971, 0.2633, 0.3504],
         ...,
         [0.5324, 0.4884, 0.5127, 0.5089, 0.6873, 0.3217],
         [0.5110, 0.4848, 0.4760, 0.5020, 0.7641, 0.7030],
         [0.5041, 0.4911, 0.4940, 0.5252, 0.3180, 0.3443]]])

In [14]:
from torch import nn
from torch.nn import functional as F

from torch.nn.attention.flex_attention import create_mask

class SimpleRoPE(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0):
        super().__init__()
        # 1. Compute inverse frequencies for half the dimension
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))

        # 2. Compute position angles: [max_seq_len, dim / 2]
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)

        # 3. Duplicate frequencies to match [max_seq_len, dim]
        emb = torch.cat((freqs, freqs), dim=-1)

        # 4. Precompute fixed cos and sin tables [1, 1, max_seq_len, dim]
        self.register_buffer("cos", emb.cos().unsqueeze(0).unsqueeze(0), persistent=False)
        self.register_buffer("sin", emb.sin().unsqueeze(0).unsqueeze(0), persistent=False)

    def _rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def forward(self, q: torch.Tensor, k: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        q, k shape: [batch_size, num_heads, seq_len, head_dim]
        """
        seq_len = q.shape[2]

        # Slice precomputed table up to current seq_len
        cos = self.cos[:, :, :seq_len, :]
        sin = self.sin[:, :, :seq_len, :]

        # Apply rotation
        q_embed = (q * cos) + (self._rotate_half(q) * sin)
        k_embed = (k * cos) + (self._rotate_half(k) * sin)

        return q_embed, k_embed

class GatedShortConv(nn.Module):
    def __init__(self, embed_dim: int, window_size: int, bias: bool = True) -> None:
        super().__init__()
        assert window_size % 2 == 1
        self.window_size = window_size
        self.in_proj = nn.Linear(embed_dim, 3 * embed_dim, bias=bias)
        self.conv = nn.Conv1d(embed_dim, embed_dim, window_size, groups=embed_dim, bias=bias)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)

    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        gate_in, gate_out, values = self.in_proj(x).chunk(3, dim=-1)
        values = (gate_in * values).transpose(1, 2)
        if causal:
            values = F.pad(values, (self.window_size - 1, 0))
        else:
            values = F.pad(values, (self.window_size // 2, self.window_size // 2))
        values = self.conv(values).transpose(1, 2)
        return self.out_proj(gate_out * values)

class HybridAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, max_seq_len: int, bias: bool = True) -> None:
        super().__init__()

        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=bias)
        self.o_proj = nn.Linear(embed_dim, embed_dim, bias=bias)

        self.rope = SimpleRoPE(self.head_dim, max_seq_len=max_seq_len)

    def _split_heads(self, x):
        return x.unflatten(-1, (self.num_heads, self.head_dim)).transpose(1, 2).contiguous()

    def forward(self, x: torch.Tensor, attn_mask):
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1) # 3x [B, N, d]

        q = self._split_heads(q)
        k = self._split_heads(k)
        v = self._split_heads(v)

        q, k = self.rope(q, k)

        attn = F.scaled_dot_product_attention(q, k , v, attn_mask=attn_mask)
        out = self.o_proj(attn.transpose(1, 2).flatten(-2, -1)) # [B, N, d]

        return out

class SwiGLU(nn.Module):
    def __init__(self, embed_dim: int, hidden_multiplier: float = 8/3, bias: bool = False) -> None:
        super().__init__()

        hidden_dim = int(embed_dim * hidden_multiplier)

        self.in_proj = nn.Linear(embed_dim, 2 * hidden_dim, bias=bias)
        self.out_proj = nn.Linear(hidden_dim, embed_dim, bias=bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate, value = self.in_proj(x).chunk(2, dim=-1)
        return self.out_proj(F.silu(gate) * value)

class HybridAttentionBlock(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, max_seq_len: int, bias: bool = True) -> None:
        super().__init__()

        self.mixer_in_norm = nn.RMSNorm(embed_dim)
        self.mixer = HybridAttention(embed_dim, num_heads, max_seq_len, bias=bias)
        self.mixer_out_norm = nn.RMSNorm(embed_dim)

        self.mlp_in_norm = nn.RMSNorm(embed_dim)
        self.mlp = SwiGLU(embed_dim, bias=bias)
        self.mlp_out_norm = nn.RMSNorm(embed_dim)

    def forward(self, x: torch.Tensor, attn_mask):
        x = x + self.mixer_out_norm(self.mixer(self.mixer_in_norm(x), attn_mask))
        x = x + self.mlp_out_norm(self.mlp(self.mlp_in_norm(x)))
        return x

class GatedShortConvBlock(nn.Module):
    def __init__(self, embed_dim: int, window_size: int, bias: bool = True) -> None:
        super().__init__()

        self.mixer_in_norm = nn.RMSNorm(embed_dim)
        self.mixer = GatedShortConv(embed_dim, window_size, bias=bias)
        self.mixer_out_norm = nn.RMSNorm(embed_dim)

        self.mlp_in_norm = nn.RMSNorm(embed_dim)
        self.mlp = SwiGLU(embed_dim, bias=bias)
        self.mlp_out_norm = nn.RMSNorm(embed_dim)

    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        x = x + self.mixer_out_norm(self.mixer(self.mixer_in_norm(x), causal))
        x = x + self.mlp_out_norm(self.mlp(self.mlp_in_norm(x)))
        return x

class StrategyNetwork(nn.Module):
    def __init__(
        self,
        in_features: int,
        embed_dim: int,
        num_heads: int,
        conv_window_size: int,
        attn_window_size: int,
        num_conv_layers: int,
        num_attn_layers: int,
        max_seq_len: int,

        target_network_out_features: int,
        target_network_embed_dim: int,
        target_network_num_heads: int,
        target_network_hidden_multiplier: float,
        target_network_conv_window_size: int,
        target_network_attn_window_size: int,
        target_network_group_size: int,
        target_network_num_conv_layers: int,
        target_network_num_attn_layers: int,

        bias: bool = True
    ) -> None:
        super().__init__()

        assert max_seq_len % target_network_group_size == 0
        assert max_seq_len // target_network_group_size == 4 * target_network_num_conv_layers + 3 * target_network_num_attn_layers

        self.prefix = nn.Parameter(torch.randn(1, 3, embed_dim))
        self.expand_layer = nn.Linear(in_features, embed_dim, bias=False)
        self.conv_blocks = nn.ModuleList([
            GatedShortConvBlock(embed_dim, conv_window_size, bias=bias) for _ in range(num_conv_layers)
        ])
        self.attn_blocks = nn.ModuleList([
            HybridAttentionBlock(embed_dim, num_heads, max_seq_len, bias=bias) for _ in range(num_attn_layers)
        ])
        self.final_norm = nn.RMSNorm(embed_dim)

        def sliding_window(b, h, q_idx, kv_idx):
            return (q_idx - kv_idx).abs() <= attn_window_size
        self.attn_mask = create_mask(sliding_window, B=None, H=None, Q_LEN=max_seq_len, KV_LEN=max_seq_len)

        self.in_proj_params = nn.Linear(embed_dim, in_features * target_network_embed_dim, bias=bias)
        self.final_norm_params = nn.Linear(embed_dim, target_network_embed_dim, bias=bias)
        self.out_proj_params = nn.Linear(embed_dim, target_network_embed_dim * target_network_out_features, bias=bias)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        x = self.expand_layer(torch.concat((self.prefix.expand_as(x), x), dim=1))
        for block in self.conv_blocks:
            x = block(x, causal=False)
        for block in self.attn_blocks:
            x = block(x, attn_mask=self.attn_mask)
        x = self.final_norm(x)

        in_proj_params = self.in_proj_params(x[:, 0, :]).mean(dim=0)
        final_norm_params = self.final_norm_params(x[:, 1, :]).mean(dim=0)
        out_proj_params = self.out_proj_params(x[:, 2, :]).mean(dim=0)

        return {
            "in_proj_params": in_proj_params,
            "final_norm_params": final_norm_params,
            "out_proj_params": out_proj_params,
        }

strategy_network = StrategyNetwork(
    in_features=6,
    embed_dim=128,
    num_heads=4,
    conv_window_size=3,
    attn_window_size=128,
    num_conv_layers=4,
    num_attn_layers=4,
    max_seq_len=dataset.past_window,

    target_network_out_features=3,
    target_network_embed_dim=128,
    target_network_num_heads=4,
    target_network_hidden_multiplier=8/3,
    target_network_conv_window_size=3,
    target_network_attn_window_size=128,
    target_network_group_size=160,
    target_network_num_conv_layers=9,
    target_network_num_attn_layers=6,

    bias=True
).to("mps")
strategy_network(next(iter(dataloader))[0].to("mps")).shape

RuntimeError: The expanded size of the tensor (6) must match the existing size (128) at non-singleton dimension 2.  Target sizes: [2, 8640, 6].  Tensor sizes: [1, 3, 128]

In [ ]:
# Non-bias model
# projection: in_features*d
# N x gated short conv block: 4d + 4d^2 + d*conv_window_size + 3*hidden_multiplier*d^2
# M x attention block: 4d + 4d^2 + 3*hidden_multiplier*d^2
# final norm: d
# projection: d*out_features